In [ ]:
import os
import json
import logging
from typing import Dict, Any, List
from data import dataset_loader

In [ ]:
logger = logging.getLogger(__name__)

In [ ]:
class UnifiedDatasetLoader:
    """Step 2: Loads JSONL dataset files recursively from data/datasets/ nested subfolders,
    with automatic fallbacks for external benchmark datasets."""

    @staticmethod
    def load_jsonl_file(file_path: str) -> List[Dict[str, Any]]:
        """Reads a single .jsonl file and returns a list of dictionaries."""
        data = []
        try:
            with open(file_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        data.append(json.loads(line))
            logger.info(f"Successfully loaded {len(data)} records from {file_path}")
        except Exception as e:
            logger.error(f"Error reading JSONL file {file_path}: {e}")
        return data

    @classmethod
    def load_local_jsonl_datasets(cls, base_dir: str = "data/datasets") -> Dict[str, List[Dict[str, Any]]]:
        """Recursively scans base_dir and its subdirectories to load all .jsonl files.
        Returns a dictionary keyed by dataset name (file stem or relative folder key)."""
        local_datasets = {}
        if not os.path.exists(base_dir):
            logger.warning(f"Directory {base_dir} does not exist. Creating default path.")
            os.makedirs(base_dir, exist_ok=True)
            return local_datasets

        for root, _, files in os.walk(base_dir):
            for file in files:
                if file.endswith(".jsonl"):
                    full_path = os.path.join(root, file)
                    # Create a key based on relative path from base_dir (e.g., "spider/train")
                    rel_dir = os.path.relpath(root, base_dir)
                    filename_no_ext = os.path.splitext(file)[0]
                    key = filename_no_ext if rel_dir == "." else f"{rel_dir}/{filename_no_ext}"

                    records = cls.load_jsonl_file(full_path)
                    local_datasets[key] = records

        return local_datasets

    @classmethod
    def load_datasets(cls, config: Dict[str, Any]) -> Dict[str, Any]:
        logger.info("Loading benchmarks and local JSONL datasets...")
        loaded_data = {}

        # 1. Load all local JSONL datasets from subfolders in data/datasets/
        base_dataset_dir = config.get("datasets", {}).get("local_dir", "data/raw)
        local_data = cls.load_local_jsonl_datasets(base_dataset_dir)
        loaded_data["local"] = local_data

        # 2. Spider Dataset (Text-to-SQL)
        try:
            loaded_data["Spider"] = load_dataset("spider", split="validation[:50]")
        except Exception:
            loaded_data["Spider"] = local_data.get("spider", [
                {"question": "Find all active users", "query": "SELECT * FROM users WHERE status = 'active';"}
            ])

        # 3. BIRD-Bench (Text-to-SQL Benchmark)
        try:
            loaded_data["BirdBench"] = load_dataset("bird_bench", split="validation[:50]")
        except Exception:
            loaded_data["BirdBench"] = local_data.get("birdbench", [
                {"question": "Highest average salary department", "query": "SELECT dept_id FROM salaries GROUP BY dept_id ORDER BY AVG(amount) DESC LIMIT 1;"}
            ])

        # 4. CoDocBench (Code-Documentation Benchmark)
        loaded_data["CoDocBench"] = local_data.get("codocbench", [
            {"code": "def calc_area(r):\n    return 3.14 * r ** 2", "docstring": "Calculates circle area."}
        ])

        return loaded_data